In [10]:
import os

# get Groq api key by reading the first line of the text file groq.txt

with open('groq.txt') as file:
    api_key = str(file.readline())

os.environ['GROQ_API_KEY'] = api_key

In [11]:
# Load CSV document
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(r'fintechdf_categorized.csv', encoding='latin1')

In [12]:
doc = loader.load()
print(f"You have {len(doc)} documents")
print(f"You have {len(doc[0].page_content)} characters in the first document")

You have 171 documents
You have 1858 characters in the first document


In [13]:
# create embeddings for all the documents

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1070.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# store embeddings in vectordatabase

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(doc, embeddings)

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
)

In [16]:
# setting up the retrieval function using modern LCEL approach

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create a simple RAG chain
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
# execute the chain

query = "Which companies are active in the field of international payments?"

result = chain.invoke(query)

In [20]:
print(result)

Based on the provided context, the following companies are active in the field of international payments:

1. PalaPlus - offers a digital payment platform designed to streamline financial transactions for businesses in Latin America, facilitating the reception of international payments.
2. viafintech - provides a payment infrastructure that enables cross-border transactions, although its primary focus is on cash-in/cash-out transactions within Europe.
3. kinoora (also known as Okoora) - offers an Automated Business Currency Management (ABCM) platform that simplifies cross-border transactions and provides FX risk management tools for businesses of all sizes.

These companies are involved in facilitating international payments, although their specific services and focus areas may differ.
